In [233]:
import warnings
warnings.filterwarnings('ignore') 
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE

import pickle
import matplotlib.pyplot as plt

In [234]:
dataset=pd.read_csv("Netflix_clean.csv")

In [235]:
dataset

,customer_id,age,gender,subscription_type,watch_hours,last_login_days,region,device,monthly_fee,churned,payment_method,number_of_profiles,avg_watch_time_per_day,favorite_genre
0,a9b75100-82a8-427a-a208-72f24052884a,51.0,Other,Basic,14.73,29.0,Africa,TV,8.99,1.0,Gift Card,1.0,29.0,Action
1,49a5dfd9-7e69-4022-a6ad-0a1b9767fb5b,47.0,Other,Standard,0.70,19.0,Europe,Mobile,13.99,1.0,Gift Card,5.0,2.0,Sci-Fi
2,4d71f6ce-fca9-4ff7-8afa-197ac24de14b,27.0,Female,Standard,16.32,10.0,Asia,TV,13.99,0.0,Crypto,2.0,89.0,Drama
3,d3c72c38-631b-4f9e-8a0e-de103cad1a7d,53.0,Other,Premium,4.51,12.0,Oceania,TV,17.99,1.0,Crypto,2.0,21.0,Horror
4,4e265c34-103a-4dbb-9553-76c9aa47e946,56.0,Other,Standard,1.89,13.0,Africa,Mobile,13.99,1.0,Crypto,2.0,8.0,Action
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4996,18779bcb-ba2b-41da-b751-e70b812061ec,67.0,Female,Basic,9.24,2.0,North America,Desktop,8.99,0.0,PayPal,3.0,185.0,Documentary
4997,3f32e8c5-615b-4a3b-a864-db2688f7834f,66.0,Male,Standard,16.55,49.0,South America,Desktop,13.99,1.0,Debit Card,2.0,20.0,Action
4998,7b0ad82d-6571-430e-90f4-906259e0e89c,59.0,Female,Basic,9.12,3.0,Europe,Laptop,8.99,0.0,Credit Card,4.0,137.0,Sci-Fi
4999,82aeef39-ddb0-40ad-bae1-5c436e0cf042,57.0,Male,Basic,1.62,17.0,Africa,Mobile,8.99,1.0,Crypto,2.0,5.0,Action


In [236]:
dataset.columns

Index(['customer_id', 'age', 'gender', 'subscription_type', 'watch_hours',
       'last_login_days', 'region', 'device', 'monthly_fee', 'churned',
       'payment_method', 'number_of_profiles', 'avg_watch_time_per_day',
       'favorite_genre'],
      dtype='object')

In [237]:
dataset.isna().sum()

customer_id               1
age                       1
gender                    1
subscription_type         1
watch_hours               1
last_login_days           1
region                    1
device                    1
monthly_fee               1
churned                   1
payment_method            1
number_of_profiles        1
avg_watch_time_per_day    1
favorite_genre            1
dtype: int64

In [238]:
dataset.isnull().sum().sum()

np.int64(14)

In [239]:
dataset.isnull().sum()[dataset.isnull().sum() > 0]

customer_id               1
age                       1
gender                    1
subscription_type         1
watch_hours               1
last_login_days           1
region                    1
device                    1
monthly_fee               1
churned                   1
payment_method            1
number_of_profiles        1
avg_watch_time_per_day    1
favorite_genre            1
dtype: int64

In [240]:
dataset[dataset.isnull().any(axis=1)]

,customer_id,age,gender,subscription_type,watch_hours,last_login_days,region,device,monthly_fee,churned,payment_method,number_of_profiles,avg_watch_time_per_day,favorite_genre
5000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [241]:
dataset = dataset.drop(index=5000)

In [242]:
dataset[dataset.isnull().any(axis=1)]

,customer_id,age,gender,subscription_type,watch_hours,last_login_days,region,device,monthly_fee,churned,payment_method,number_of_profiles,avg_watch_time_per_day,favorite_genre


In [243]:
dataset=pd.get_dummies(dataset,dtype=int,drop_first=True)

In [244]:
dataset.columns

Index(['age', 'watch_hours', 'last_login_days', 'monthly_fee', 'churned',
       'number_of_profiles', 'avg_watch_time_per_day',
       'customer_id_00139d09-c969-4136-987c-6e7d30f41cc0',
       'customer_id_0017c041-a14b-4780-9d9a-7a99558a0768',
       'customer_id_001b5926-1113-493d-80a4-f09012f2f43b',
       ...
       'payment_method_Crypto', 'payment_method_Debit Card',
       'payment_method_Gift Card', 'payment_method_PayPal',
       'favorite_genre_Comedy', 'favorite_genre_Documentary',
       'favorite_genre_Drama', 'favorite_genre_Horror',
       'favorite_genre_Romance', 'favorite_genre_Sci-Fi'],
      dtype='object', length=5029)

In [245]:
print([col for col in dataset.columns if col.startswith("subscription_type_")])

['subscription_type_Premium', 'subscription_type_Standard']


In [246]:
independent=dataset[['age', 'watch_hours', 'last_login_days', 'monthly_fee', 'subscription_type_Premium', 'subscription_type_Standard', 'number_of_profiles', 'avg_watch_time_per_day',
       'payment_method_Crypto', 'payment_method_Debit Card',
       'payment_method_Gift Card', 'payment_method_PayPal',
       'favorite_genre_Comedy', 'favorite_genre_Documentary',
       'favorite_genre_Drama', 'favorite_genre_Horror',
       'favorite_genre_Romance', 'favorite_genre_Sci-Fi']]

In [247]:
independent

,age,watch_hours,last_login_days,monthly_fee,subscription_type_Premium,subscription_type_Standard,number_of_profiles,avg_watch_time_per_day,payment_method_Crypto,payment_method_Debit Card,payment_method_Gift Card,payment_method_PayPal,favorite_genre_Comedy,favorite_genre_Documentary,favorite_genre_Drama,favorite_genre_Horror,favorite_genre_Romance,favorite_genre_Sci-Fi
0,51.0,14.73,29.0,8.99,0,0,1.0,29.0,0,0,1,0,0,0,0,0,0,0
1,47.0,0.70,19.0,13.99,0,1,5.0,2.0,0,0,1,0,0,0,0,0,0,1
2,27.0,16.32,10.0,13.99,0,1,2.0,89.0,1,0,0,0,0,0,1,0,0,0
3,53.0,4.51,12.0,17.99,1,0,2.0,21.0,1,0,0,0,0,0,0,1,0,0
4,56.0,1.89,13.0,13.99,0,1,2.0,8.0,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,19.0,49.17,11.0,8.99,0,0,4.0,246.0,0,0,0,0,0,0,1,0,0,0
4996,67.0,9.24,2.0,8.99,0,0,3.0,185.0,0,0,0,1,0,1,0,0,0,0
4997,66.0,16.55,49.0,13.99,0,1,2.0,20.0,0,1,0,0,0,0,0,0,0,0
4998,59.0,9.12,3.0,8.99,0,0,4.0,137.0,0,0,0,0,0,0,0,0,0,1


In [248]:
dependent=dataset[['churned']]

In [249]:
dependent

,churned
0,1.0
1,1.0
2,0.0
3,1.0
4,1.0
...,...
4995,0.0
4996,0.0
4997,1.0
4998,0.0


In [250]:
dataset

,age,watch_hours,last_login_days,monthly_fee,churned,number_of_profiles,avg_watch_time_per_day,customer_id_00139d09-c969-4136-987c-6e7d30f41cc0,customer_id_0017c041-a14b-4780-9d9a-7a99558a0768,customer_id_001b5926-1113-493d-80a4-f09012f2f43b,...,payment_method_Crypto,payment_method_Debit Card,payment_method_Gift Card,payment_method_PayPal,favorite_genre_Comedy,favorite_genre_Documentary,favorite_genre_Drama,favorite_genre_Horror,favorite_genre_Romance,favorite_genre_Sci-Fi
0,51.0,14.73,29.0,8.99,1.0,1.0,29.0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
1,47.0,0.70,19.0,13.99,1.0,5.0,2.0,0,0,0,...,0,0,1,0,0,0,0,0,0,1
2,27.0,16.32,10.0,13.99,0.0,2.0,89.0,0,0,0,...,1,0,0,0,0,0,1,0,0,0
3,53.0,4.51,12.0,17.99,1.0,2.0,21.0,0,0,0,...,1,0,0,0,0,0,0,1,0,0
4,56.0,1.89,13.0,13.99,1.0,2.0,8.0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,19.0,49.17,11.0,8.99,0.0,4.0,246.0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
4996,67.0,9.24,2.0,8.99,0.0,3.0,185.0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
4997,66.0,16.55,49.0,13.99,1.0,2.0,20.0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
4998,59.0,9.12,3.0,8.99,0.0,4.0,137.0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
def split_scalar(independent,dependent):
    x_train,x_test,y_train,y_test=train_test_split(independent,dependent,test_size=0.3,random_state=0)
    sc=StandardScaler()
    x_train=sc.fit_transform(x_train)
    x_test=sc.transform(x_test)
    return x_train,x_test,y_train,y_test


In [251]:
from sklearn.feature_selection import SelectKBest, chi2
def selectk_best(independent,dependent,n):
    
    k_best=SelectKBest(score_func=chi2,k=n)
    fit1=k_best.fit(independent,dependent)
    select_k=fit1.transform(independent)
    return select_k

In [252]:
def cm_prediction(classifier,x_test,y_test):
    y_pred=classifier.predict(x_test)
    from sklearn.metrics import confusion_matrix
    cm=confusion_matrix(y_test,y_pred)
    from sklearn.metrics import accuracy_score
    from sklearn.metrics import classification_report
    accuracy=accuracy_score(y_test,y_pred)
    report=classification_report(y_test,y_pred)
    return classifier,accuracy,report,cm,x_test,y_test


In [253]:
from sklearn.linear_model import LogisticRegression
def logistic(x_train,y_train,x_test,y_test):
    classifier=LogisticRegression()
    classifier.fit(x_train,y_train)
    classifier,accuracy,report,cm,x_test,y_test=cm_prediction(classifier,x_test,y_test)
    return classifier,accuracy,report,cm,x_test,y_test


In [254]:
from sklearn.svm import SVC
def SVCL(x_train,y_train,x_test,y_test):
    classifier=SVC(kernel = 'linear', random_state = 0)
    classifier.fit(x_train,y_train)
    classifier,accuracy,report,cm,x_test,y_test=cm_prediction(classifier,x_test,y_test)
    return classifier,accuracy,report,cm,x_test,y_test

               

In [255]:
from sklearn.svm import SVC
def SVCNL(x_train,y_train,x_test,y_test):
    classifier=SVC(kernel = 'rbf', random_state = 0)
    classifier.fit(x_train,y_train)
    classifier,accuracy,report,cm,x_test,y_test=cm_prediction(classifier,x_test,y_test)
    return classifier,accuracy,report,cm,x_test,y_test

In [256]:
from sklearn.ensemble import RandomForestClassifier
def rf(x_train,y_train,x_test,y_test):
    classifier=RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
    classifier.fit(x_train,y_train)
    classifier,accuracy,report,cm,x_test,y_test=cm_prediction(classifier,x_test,y_test)
    return classifier,accuracy,report,cm,x_test,y_test

In [257]:
from sklearn.tree import DecisionTreeClassifier
def dt(x_train,y_train,x_test,y_test):
    classifier=DecisionTreeClassifier(criterion='entropy',random_state = 0)
    classifier.fit(x_train,y_train)
    classifier,accuracy,report,cm,x_test,y_test=cm_prediction(classifier,x_test,y_test)
    return classifier,accuracy,report,cm,x_test,y_test

In [258]:
from sklearn.naive_bayes import GaussianNB   
               
def Navie(x_train,y_train,x_test,y_test):  
    classifier = GaussianNB()
    classifier.fit(x_train, y_train)
    classifier,accuracy,report,cm,x_test,y_test=cm_prediction(classifier,x_test,y_test)
    return  classifier, accuracy, report, cm, x_test, y_test           
        


In [259]:
from sklearn.neighbors import KNeighborsClassifier              
def knn(x_train,y_train,x_test,y_test): 
    classifier = KNeighborsClassifier(n_neighbors = 5, metric = 'minkowski', p = 2)
    classifier.fit(x_train, y_train)
    classifier,accuracy,report,cm,x_test,y_test=cm_prediction(classifier,x_test,y_test)
    return  classifier, accuracy, report, cm, x_test, y_test         

In [260]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
def ada_boost(x_train,y_train,x_test,y_test):
    classifier=AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=3),n_estimators=100,learning_rate=0.1)
    classifier.fit(x_train, y_train)
    classifier,accuracy,report,cm,x_test,y_test=cm_prediction(classifier,x_test,y_test)
    return  classifier, accuracy, report, cm, x_test, y_test  

In [261]:
#!pip install catboost

In [262]:
from catboost import CatBoostClassifier
def cat_boost(x_train,y_train,x_test,y_test):
    classifier=CatBoostClassifier(iterations=5,learning_rate=0.1, )
    classifier.fit(x_train, y_train)
    classifier,accuracy,report,cm,x_test,y_test=cm_prediction(classifier,x_test,y_test)
    return  classifier, accuracy, report, cm, x_test, y_test 

In [263]:
from xgboost import XGBClassifier
def xgb(x_train,y_train,x_test,y_test):
    classifier=XGBClassifier(n_estimators=100,learning_rate=0.05,max_depth=5)
    classifier.fit(x_train,y_train)
    classifier,accuracy,report,cm,x_test,y_test=cm_prediction(classifier,x_test,y_test)
    return  classifier, accuracy, report, cm, x_test, y_test 

In [264]:
def selectk_Classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf,accadaboost,acccatboost,accxgb): 
    
    dataframe=pd.DataFrame(index=['ChiSquare'],columns=['Logistic','SVCL','SVCNL','KNN','Navie','Decision','Random','AdaBoost','CatBoost','xgb'])
    for number,index in enumerate(dataframe.index):      
        dataframe['Logistic'][index]=acclog[number]       
        dataframe['SVCL'][index]=accsvml[number]
        dataframe['SVCNL'][index]=accsvmnl[number]
        dataframe['KNN'][index]=accknn[number]
        dataframe['Navie'][index]=accnav[number]
        dataframe['Decision'][index]=accdes[number]
        dataframe['Random'][index]=accrf[number]
        dataframe['AdaBoost'][index]=accadaboost[number]
        dataframe['CatBoost'][index]=acccatboost[number]
        dataframe['xgb'][index]=accxgb[number]
    return dataframe

In [276]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

acclog=[]
accsvml=[]
accsvmnl=[]
accknn=[]
accnav=[]
accdes=[]
accrf=[]
accadaboost=[]
acccatboost=[]
accxgb=[]

# Split the dataset
x_train, x_test, y_train, y_test = train_test_split(
    independent, dependent, test_size=0.3, random_state=0
)

# Ask user how many features they want
k = int(input("Enter number of best features: "))

# Select the best features
k_best = SelectKBest(score_func=chi2, k=k)

fit1 = k_best.fit(x_train, y_train)

x_train = fit1.transform(x_train)
x_test = fit1.transform(x_test)

# Show selected feature names
best_features = independent.columns[k_best.get_support()]
display(pd.DataFrame({"Selected Features": best_features}))

# Scale the selected features
sc = StandardScaler()

x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)


# 1. Logistic Regression
log_classifier, log_accuracy, log_report, log_cm, x_test, y_test = logistic(x_train,y_train,x_test,y_test)
acclog.append(log_accuracy)

# 2. Linear SVM
svml_classifier, svml_accuracy, svml_report, svml_cm, x_test, y_test = SVCL(x_train,y_train,x_test,y_test)
accsvml.append(svml_accuracy)

# 3. Non-Linear SVM
svmnl_classifier, svmnl_accuracy, svmnl_report, svmnl_cm, x_test, y_test = SVCNL(x_train,y_train,x_test,y_test)
accsvmnl.append(svmnl_accuracy)

# 4. KNN
knn_classifier, knn_accuracy, knn_report, knn_cm, x_test, y_test = knn(x_train,y_train,x_test,y_test)
accknn.append(knn_accuracy)

# 5. Naive Bayes
nav_classifier, nav_accuracy, nav_report, nav_cm, x_test, y_test = Navie(x_train,y_train,x_test,y_test)
accnav.append(nav_accuracy)

# 6. Decision Tree
dt_classifier, dt_accuracy, dt_report, dt_cm, x_test, y_test = dt(x_train,y_train,x_test,y_test)
accdes.append(dt_accuracy)

# 7. Random Forest
rf_classifier, rf_accuracy, rf_report, rf_cm, x_test, y_test = rf(x_train,y_train,x_test,y_test)
accrf.append(rf_accuracy)

# 8. AdaBoost
ada_classifier, ada_accuracy, ada_report, ada_cm, x_test, y_test = ada_boost(x_train,y_train,x_test,y_test)
accadaboost.append(ada_accuracy)

# 9. CatBoost
cat_classifier, cat_accuracy, cat_report, cat_cm, x_test, y_test = cat_boost(x_train,y_train,x_test,y_test)
acccatboost.append(cat_accuracy)

# 10. XGBoost
xgb_classifier, xgb_accuracy, xgb_report, xgb_cm, x_test, y_test = xgb(x_train,y_train,x_test,y_test)
accxgb.append(xgb_accuracy)


# Compare all algorithms
result = selectk_Classification(
    acclog, accsvml, accsvmnl, accknn, accnav,
    accdes, accrf, accadaboost, acccatboost, accxgb
)

Rank = result.T

Rank["Rank"] = Rank["ChiSquare"].rank(ascending=False)

Rank = Rank.sort_values("ChiSquare", ascending=False)

display(Rank)

Enter number of best features:  8


,Selected Features
0,watch_hours
1,last_login_days
2,monthly_fee
3,subscription_type_Premium
4,number_of_profiles
5,avg_watch_time_per_day
6,payment_method_Crypto
7,payment_method_Gift Card


0:	learn: 0.6366611	total: 13.2ms	remaining: 52.9ms
1:	learn: 0.5838093	total: 15.7ms	remaining: 23.6ms
2:	learn: 0.5362686	total: 17.4ms	remaining: 11.6ms
3:	learn: 0.4980057	total: 19.9ms	remaining: 4.98ms
4:	learn: 0.4638882	total: 21.7ms	remaining: 0us


,ChiSquare,Rank
xgb,0.974,1.0
Random,0.968,2.0
AdaBoost,0.964,3.0
Decision,0.957333,4.0
KNN,0.922667,5.0
CatBoost,0.914667,6.0
SVCNL,0.908,7.0
SVCL,0.892,8.0
Logistic,0.89,9.0
Navie,0.822667,10.0


In [221]:
best_model = xgb_classifier

In [222]:
import pickle

pickle.dump(best_model,open("xgboost_model.pkl","wb"))
pickle.dump(sc,open("scaler.pkl","wb"))

print("Files saved successfully")

Files saved successfully


In [223]:

watch_hours = float(input("Enter total watch hours: "))

if watch_hours >= 0:
    print("Valid watch hours")
else:
    print("Invalid watch hours")
    watch_hours = float(input("Please enter valid watch hours: "))


last_login_days = int(input("Enter number of days since last login: "))

if last_login_days >= 0:
    print("Valid last login days")
else:
    print("Invalid last login days")
    last_login_days = int(input("Please enter valid number of days: "))


monthly_fee = float(input("Enter monthly subscription fee [8.99, 13.99, 17.99]: "))

if monthly_fee == 8.99 or monthly_fee == 13.99 or monthly_fee == 17.99:
    print("Valid monthly fee")
else:
    print("Invalid monthly fee")
    monthly_fee = float(input("Please enter 8.99, 13.99 or 17.99: "))


subscription_type = input("Enter subscription type (Basic/Standard/Premium): ").lower()

if subscription_type == "basic" or subscription_type == "standard" or subscription_type == "premium":
    print("Valid subscription type")
else:
    print("Invalid subscription type")
    subscription_type = input("Please enter Basic/Standard/Premium: ").lower()

if subscription_type == "premium":
    subscription_type_Premium = 1
else:
    subscription_type_Premium = 0


number_of_profiles = int(input("Enter number of profiles (1 to 5): "))

if number_of_profiles >= 1 and number_of_profiles <= 5:
    print("Valid number of profiles")
else:
    print("Invalid number of profiles")
    number_of_profiles = int(input("Please enter a number between 1 and 5: "))


avg_watch_time_per_day = float(input("Enter average watch time per day in mins: "))

if avg_watch_time_per_day >= 0:
    print("Valid average watch time")
else:
    print("Invalid average watch time")
    avg_watch_time_per_day = float(input("Please enter valid watch time in mins: "))


payment_method = input("Enter payment method (Credit/Debit/Giftcard/Crypto/Paypal): ").lower()

if payment_method == "credit" or payment_method == "debit" or payment_method == "giftcard" or payment_method == "crypto" or payment_method == "paypal":
    print("Valid payment method")
else:
    print("Invalid payment method")
    payment_method = input("Please enter Credit/Debit/Giftcard/Crypto/Paypal: ").lower()

if payment_method == "crypto":
    payment_method_Crypto = 1
else:
    payment_method_Crypto = 0


Enter total watch hours:  7


Valid watch hours


Enter number of days since last login:  56


Valid last login days


Enter monthly subscription fee [8.99, 13.99, 17.99]:  17.99


...

Valid number of profiles


Enter average watch time per day in mins:  10


Valid average watch time


Enter payment method (Credit/Debit/Giftcard/Crypto/Paypal):  debit


Valid payment method


In [224]:
customer = [[
    watch_hours,
    last_login_days,
    monthly_fee,
    subscription_type_Premium,
    number_of_profiles,
    avg_watch_time_per_day,
    payment_method_Crypto
]]

print(customer)

[[7.0, 56, 17.99, 1, 4, 10.0, 0]]


In [225]:
customer_scaled = sc.transform(customer)

print(customer_scaled)

[[-0.3888179   1.50115734  1.16463507  1.39414853  0.69534502 -0.28897699
  -0.48252302]]


In [227]:
prediction = best_model.predict(customer_scaled)

if prediction==1:
    print("Customer is likely to churn")
else:
    print("Customer is unlikely to churn")



Customer is unlikely to churn


In [219]:
xgb_classifier, xgb_accuracy, xgb_report, xgb_cm, x_test, y_test = xgb(x_train, y_train, x_test, y_test)
print("\nXGBoost Accuracy:", xgb_accuracy)
print("XGBoost CM:")
print(xgb_cm)


XGBoost Accuracy: 0.964
XGBoost CM:
[[743  19]
 [ 35 703]]
